In [7]:
#Best First Search
from queue import PriorityQueue


graph = {
    'S': [('A', 3), ('B', 6), ('C', 5)],
    'A': [('D', 9), ('E', 8)],
    'B': [('F', 12), ('G', 14)],
    'C': [('H', 7)],
    'H': [('I', 5), ('J', 6)],
    'I': [('K', 1), ('L', 10), ('M', 2)],
    'D': [], 'E': [],
    'F': [], 'G': [],
    'J': [], 'K': [],
    'L': [], 'M': []
}

def best_first_search(graph,start,goal):
    visited = set()
    pq = PriorityQueue()
    pq.put((0,start))
    totalCost=0

    while not pq.empty():
        cost, node = pq.get()
        if node in visited:
            continue

        print(node + " -> ",end='')
        visited.add(node)
        totalCost+=cost
        if node == goal:
            print("\nGoal Reached")
            print("Path Cost = ",totalCost)
            return
        
        for neighbour, weight in graph[node]:
            if neighbour not in visited:
                pq.put((weight,neighbour))
    print("Goal Not Found")


best_first_search(graph,'S','I')

S -> A -> C -> B -> H -> I -> 
Goal Reached
Path Cost =  26


In [13]:
#Greedy
graph = {
    'A': {'B': 2, 'C': 1},
    'B': {'D': 4, 'E': 3},
    'C': {'F': 1, 'G': 5},
    'D': {'H': 2},
    'E': {},
    'F': {'I': 6},
    'G': {},
    'H': {},
    'I': {}
}

heuristic = {
    'A': 7, 'B': 6, 'C': 5,
    'D': 4, 'E': 7, 'F': 3,
    'G': 6, 'H': 2, 'I': 0
}

def greedy(graph,start,goal):
    frontier = [(start, heuristic[start])] #List-based PQ
    visited = set()
    came_from = {start: None}    #for reconstruction

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, _ = frontier.pop(0)   #returns node with best heuristic

        if current_node in visited:
            continue
        print(current_node + " -> ", end=" ")
        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()

            print("Goal Found")
            print("\nPath - ", path)
            return
        
        for neighbour in graph[current_node]:
            if neighbour not in visited:
                came_from[neighbour] = current_node
                frontier.append((neighbour,heuristic[neighbour]))
    print("Goal Not Found")

greedy(graph,'A','I')


A ->  C ->  F ->  I ->  Goal Found

Path -  ['A', 'C', 'F', 'I']


In [16]:
#maze-solver
from queue import PriorityQueue
maze = [
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 1],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0]
]

start = (0, 0)
end = (4, 4)

class Node:
    def __init__(self,position,parent=None):
        self.position = position
        self.parent = parent
        self.g = 0  #cost from start to current node
        self.h = 0  #heuristic estimate of the cost from current to end node
        self.f = 0  #total cost

    def __lt__(self,other):
        return self.f < other.f
    #this was like operator overloading of <

def heuristic(current_pos, end_pos):
    #manhattan distance
    return abs(current_pos[0] - end_pos[0]) + abs(current_pos[1] - end_pos[1])

def best_first_search(maze,start,end):
    rows,cols = len(maze), len(maze[0])
    start_node = Node(start)
    end_node = Node(end)
    frontier = PriorityQueue()
    frontier.put(start_node)
    visited = set()

    while not frontier.empty():
        current_node = frontier.get()
        current_pos = current_node.position
        if current_pos == end:
            path = []
            while current_node:
                path.append(current_node.position)
                current_node = current_node.parent
            path.reverse()
            return path
            #both steps can be done like return path[::-1]
    
        visited.add(current_pos)
        #since neighbours are in 4 positions i.e (1,0)(0,1)(-1,0)(0,-1) from current node's perspective
        for dx,dy in [(1,0),(-1,0),(0,1),(0,-1)]:
            new_pos = (current_pos[0]+dx , current_pos[1]+dy)
            if new_pos[0] < 0 or new_pos[1] < 0 or new_pos[0] >= rows or new_pos[1] >= cols or new_pos in visited:  #out of bound or visit check
                continue
            if maze[new_pos[0]][new_pos[1]] == 1:   #check if wall or empty
                continue

            new_node = Node(new_pos,current_node)
            new_node.g = current_node.g + 1
            new_node.h = heuristic(new_pos,end)
            new_node.f = new_node.h #bcz in best-first: f(n)=h(n)

            frontier.put(new_node)
            visited.add(new_pos)
    return None

path = best_first_search(maze,start,end)
print("Path Found - ", path)


Path Found -  [(0, 0), (1, 0), (1, 1), (1, 2), (1, 3), (2, 3), (3, 3), (3, 4), (4, 4)]


In [19]:
#A* algorithm
from queue import PriorityQueue
graph = {
'A': {'B': 4, 'C': 3},
'B': {'E': 12, 'F': 5},
'C': {'D': 7, 'E': 10},
'D': {'E': 2},
'E': {'G': 5},
'F': {'G': 16},
'G': {},
}
heuristic = {'A': 14,'B': 12,'C': 11,'D': 6,'E': 4, 'F': 11,'G': 0 }

def a_star(graph, start, goal):
    frontier = [(start, 0+heuristic[start])]
    g_costs = {start:0}
    came_from = {start: None}
    visited = set()

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, current_f = frontier.pop(0)

        if current_node in visited:
            continue
        print(current_node,end = " ")
        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print("Goal Found, Path - ", path)
            return
        
        for neighbour,cost in graph[current_node].items():
            new_gCost = g_costs[current_node] + cost
            f_cost = new_gCost + heuristic[neighbour]


            if neighbour not in g_costs or new_gCost < g_costs[neighbour]:
                g_costs[neighbour] = new_gCost
                came_from[neighbour] = current_node
                frontier.append((neighbour, f_cost))
    print("Goal Not Found")

a_star(graph,'A','G')

A C B D E G Goal Found, Path -  ['A', 'C', 'D', 'E', 'G']
